In [1]:
import Pkg

function find_flexopt_root(start_dir=joinpath(homedir(), "flexOPT"))
    candidates = String[]

    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end

    directory = abspath(expanduser(start_dir))

    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end

    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_file = joinpath(candidate, "src", "commonBatchs.jl")

        if isfile(project_file) && isfile(source_file)
            return candidate
        end
    end

    error("Cannot locate flexOPT. Set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)

@show VERSION Threads.nthreads() Base.active_project()

  Activating 

VERSION = v"1.12.6"
Threads.nthreads() = 1
Base.active_project() = "/home/ydenizhernandez/flexOPT/Project.toml"


project at `~/flexOPT`


"/home/ydenizhernandez/flexOPT/Project.toml"

In [2]:
Pkg.add("IJulia")
Pkg.add("PythonCall")
ENV["JULIA_PYTHONCALL_EXE"] = "/usr/bin/python3" #Where should the user look for this?
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"

using PythonCall
pyimport("sys").executable

   Resolving package versions...
     Project No packages added to or removed from `~/flexOPT/Project.toml`
    Manifest No packages added to or removed from `~/flexOPT/Manifest.toml`
Precompiling packages...
   9591.6 ms  ✓ InlineStrings → ParsersExt
   1139.1 ms  ✓ WeakRefStrings
  14535.4 ms  ✓ Conda
  14695.2 ms  ✓ ColorBrewer
  19767.1 ms  ✓ BenchmarkTools
  15888.5 ms  ✓ CSV
  42990.0 ms  ✓ SciMLBase → SciMLBasePythonCallExt
  65556.5 ms  ✓ Plots
   2823.7 ms  ✓ Plots → IJuliaExt
   3043.0 ms  ✓ Plots → FileIOExt
   3080.9 ms  ✓ Plots → GeometryBasicsExt
   3090.6 ms  ✓ Plots → UnitfulExt
   4041.4 ms  ✓ StatGeochem → StatGeochemPlotsExt
 116451.5 ms  ✓ Makie
   5105.2 ms  ✓ DomainSets → DomainSetsMakieExt
   5332.1 ms  ✓ SciMLBase → SciMLBaseMakieExt
  33310.7 ms  ✓ CairoMakie
  63449.9 ms  ✓ GLMakie
  18 dependencies successfully precompiled in 198 seconds. 687 already precompiled.
   Resolving package versions...
     Project No packages added to or removed from `~/flexOPT/Pro

Python: '/usr/bin/python3'

In [3]:
os = pyimport("os")
sys = pyimport("sys")

ROOT_LIB = "/home/ydenizhernandez/root-install/lib" #Should be the path to the root library, not the root source code. This is where the ROOT.so file is located.
PROJECT_ROOT = "/home/ydenizhernandez/ModularEarth" # Shoukd be the path to the root source code, where the CMakeLists.txt file is located.

# Match the configuration of your working Python notebook
sys.path.insert(0, ROOT_LIB)
os.chdir(PROJECT_ROOT)

# Attempt to import CERN ROOT
ROOT = pyimport("ROOT")

println("ROOT version: ", ROOT.gROOT.GetVersion())

ROOT version: 6.36.000


In [4]:
# ── Step 1: Load the compiled ModularEarth shared library ──────────────────
load_status = ROOT.gSystem.Load("./libModularEarth.so")
# 0 = loaded successfully, 1 = already loaded, negative = error
println("Library load status : ", pyconvert(Int, load_status),
        "  (0 = success, 1 = already loaded)")

ME = ROOT.ModularEarth
println("ModularEarth namespace bound : ", ME)

Library load status : 0  (0 = success, 1 = already loaded)
ModularEarth namespace bound : <namespace cppyy.gbl.ModularEarth at 0x3e2444d0>


In [5]:
# ── Step 2: Construct Earth and load a PREM table ───────────────────────────
# prem_15layers is the smallest available table — builds quickly
earth = ME.Earth("JuliaTest", "prem_15layers", 60.0, 60.0)
println("Earth object created")

# Populate layersconfigs from the PREM table
earth.SetEarthModel("prem_15layers")

println("\n--- Model Summary ---")
earth.GetModelSummary()

Earth object created


Info in <TGeoManager::TGeoManager>: Geometry JuliaTest, JuliaTest Earth Model created
Info in <TGeoManager::SetTopVolume>: Top volume is EarthTopSphere. Master volume is EarthTopSphere
Info in <TGeoNavigator::BuildCache>: --- Maximum geometry depth set to 100



--- Model Summary ---

--- Layer index: 0Index check: 1 ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer index: 1Index check: 2 ---
  Sublayer 1 | rmin: 1221.5 | rmax: 1946.7 | ρ: 11.9512 | Z/A: 0.4691
  Sublayer 2 | rmin: 1946.7 | rmax: 2475.1 | ρ: 11.4726 | Z/A: 0.4691
  Sublayer 3 | rmin: 2475.1 | rmax: 2900.3 | ρ: 10.9884 | Z/A: 0.4691
  Sublayer 4 | rmin: 2900.3 | rmax: 3260.6 | ρ: 10.5008 | Z/A: 0.4691
  Sublayer 5 | rmin: 3260.6 | rmax: 3480 | ρ: 10.0861 | Z/A: 0.4691

--- Layer index: 2Index check: 3 ---
  Sublayer 1 | rmin: 3480 | rmax: 4476 | ρ: 5.3225 | Z/A: 0.4954
  Sublayer 2 | rmin: 4476 | rmax: 5378.4 | ρ: 4.8332 | Z/A: 0.4954
  Sublayer 3 | rmin: 5378.4 | rmax: 5701 | ρ: 4.4872 | Z/A: 0.4954

--- Layer index: 3Index check: 4 ---
  Sublayer 1 | rmin: 5701 | rmax: 5971 | ρ: 3.8983 | Z/A: 0.4954
  Sublayer 2 | rmin: 5971 | rmax: 6346.6 | ρ: 3.4306 | Z/A: 0.4954

--- Layer index: 4Index check: 5 ---
  Sublayer 1 | rmin: 6346.6 | rmax: 6356 | 

Python: None

In [ ]:
# ── Test: SetSublayerUpdates — pre-build config check ──────────────────────
# SetSublayerUpdates must be called BEFORE BuiltEarth().
# GetModelSummary() reads directly from layersconfigs, so comparing its output
# before and after the call is the ground-truth check that the config was mutated.
#
# Target  : Layer 0 (Inner Core), Sublayer 0
# Original: ρ = 12.9807 g/cm³,  Z/A = 0.4691
# Updated : ρ = 10.0   g/cm³,  Z/A = 0.45

test_ly   = 0       # Inner Core — major layer index
test_sly  = 0       # first sublayer of the Inner Core
orig_rho  = 12.9807
orig_ye   = 0.4691
new_rho   = 10.0
new_ye    = 0.45
new_name  = "TestMaterial_InnerCore"

println("─── BEFORE SetSublayerUpdates ─────────────────────────────────────────")
earth.GetModelSummary()

earth.SetSublayerUpdates(test_ly, test_sly, new_rho, new_ye, new_name)

println("\n─── AFTER SetSublayerUpdates ──────────────────────────────────────────")
earth.GetModelSummary()
println("\nExpected Layer $test_ly Sublayer $test_sly: ρ = $new_rho  Z/A = $new_ye  name = $new_name")

In [6]:
# ── Step 3: Build the geometry tree and close it ────────────────────────────
# BuiltEarth() constructs Layer/Sublayer volumes, adds them to the top
# volume, then calls CloseGeometry() through the compiled C++ wrapper.
earth.BuiltEarth()
println("BuiltEarth() returned")

TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material SublayerMaterial defined with fractional Z=0.469100
TGeoMateri

BuiltEarth() returned


In [7]:
# ── Step 4: Verify geometry finalization ────────────────────────────────────
geo = earth.GetGeoManager()
top = earth.GetTopVolume()

geo_name = pyconvert(String, geo.GetName())
top_name = pyconvert(String, top.GetName())
is_closed = pyconvert(Bool, geo.IsClosed())
n_daughters = pyconvert(Int, top.GetNdaughters())

println("TGeoManager name   : ", geo_name)
println("Top volume name    : ", top_name)
println("IsClosed           : ", is_closed)
println("Top daughters      : ", n_daughters, " (one per major layer)")

# Cross-check: ROOT keeps a global gGeoManager; it should be Earth's manager
global_geo = ROOT.gGeoManager
global_name = pyconvert(String, global_geo.GetName())
println("\nglobal gGeoManager : ", global_name)
println("Same as Earth's?   : ", geo_name == global_name)

@assert is_closed "FAIL: geometry was not closed"
@assert n_daughters > 0 "FAIL: top volume has no daughters"
println("\nAll assertions passed.")

TGeoManager name   : JuliaTest
Top volume name    : EarthTopSphere
IsClosed           : true
Top daughters      : 7 (one per major layer)

global gGeoManager : JuliaTest
Same as Earth's?   : true

All assertions passed.


In [ ]:
# ── Test: SetSublayerUpdates — post-build geometry assertion ───────────────
# BuiltEarth() constructed the ROOT geometry from the mutated layersconfigs.
# Navigate to (0, 0, 0) — inside Inner Core sublayer 0 — and confirm that
# the material density in the live ROOT geometry matches the value written
# by SetSublayerUpdates (new_rho = 10.0 g/cm³).

geo = earth.GetGeoManager()

node_at_origin = geo.FindNode(0.0, 0.0, 0.0)
mat            = geo.GetCurrentVolume().GetMedium().GetMaterial()

mat_name    = pyconvert(String,  mat.GetName())
mat_density = pyconvert(Float64, mat.GetDensity())

println("Node at (0,0,0)  : ", pyconvert(String, node_at_origin.GetName()))
println("Material name    : ", mat_name)
println("Material density : ", mat_density, " g/cm³")
println("Expected density : ", new_rho,     " g/cm³")
println("Expected name    : ", new_name)

@assert abs(mat_density - new_rho) < 1e-6 \
    "FAIL: geometry density ($mat_density) does not match SetSublayerUpdates value ($new_rho)"

@assert mat_name == new_name \
    "FAIL: material name ($mat_name) does not match SetSublayerUpdates value ($new_name)"

println("\nBoth assertions passed — SetSublayerUpdates propagated correctly to ROOT geometry.")

In [10]:
# ── Step 5: Smoke-test geometry navigation ──────────────────────────────────
# Navigate to the master volume; this fails if the geometry is not closed.
geo.CdTop()
current_vol = pyconvert(String, geo.GetCurrentVolume().GetName())
println("Current volume after cd(\"/\") : ", current_vol)

# FindNode is a basic navigation sanity check
found = geo.FindNode(0.0, 0.0, 0.0)   # centre of the Earth
node_name = pyconvert(String, found.GetName())
println("Node at (0,0,0)              : ", node_name)

println("\nGeometry is closed and navigable from Julia via PythonCall/PyROOT.")

Current volume after cd("/") : EarthTopSphere
Node at (0,0,0)              : BlockVolume_1_1

Geometry is closed and navigable from Julia via PythonCall/PyROOT.
